In [2]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar

def shooting(ode, u0, beta):
    """
    A function that uses numerical shooting to find limit cycles of
    a specified ODE.

    Parameters
    ----------
    ode : function
        The ODE to apply shooting to. The ode function should take
        a single parameter (the state vector) and return the
        right-hand side of the ODE as a numpy.array.
    u0 : numpy.array
        An initial guess at the initial values for the limit cycle.
    beta : float
        The parameter β in the ODE.

    Returns
    -------
    Returns a numpy.array containing the corrected initial values
    for the limit cycle. If the numerical root finder failed, the
    returned array is empty.
    """
    def residual(theta):
        u1_guess = np.sqrt(beta) * np.cos(theta)
        u2_guess = np.sqrt(beta) * np.sin(theta)
        y = ode(np.array([u1_guess, u2_guess]))
        return np.array([y[0] - u1_guess, y[1] - u2_guess])
    
    result = root_scalar(residual, bracket=[0, 2 * np.pi])
    if result.converged:
        theta = result.root
        u1_cycle = np.sqrt(beta) * np.cos(theta)
        u2_cycle = np.sqrt(beta) * np.sin(theta)
        return np.array([u1_cycle, u2_cycle])
    else:
        return np.array([])

# Example ODE for Hopf bifurcation normal form
def hopf_normal_form(u, beta):
    u1, u2 = u
    return np.array([
        beta * u1 - u2 - u1 * (u1 ** 2 + u2 ** 2),
        u1 + beta * u2 - u2 * (u1 ** 2 + u2 ** 2)
    ])

# Example test function
def test_shooting_hopf_normal_form():
    # Test with known solution for β = 1
    u0_guess = np.array([0.5, 0.5])
    u_cycle = shooting(hopf_normal_form, u0_guess, 1)
    u_cycle_expected = np.array([np.sqrt(1) * np.cos(np.pi / 4), np.sqrt(1) * np.sin(np.pi / 4)])
    assert np.allclose(u_cycle, u_cycle_expected), "Failed test_shooting_hopf_normal_form"

# Run tests
if __name__ == "__main__":
    test_shooting_hopf_normal_form()
    print("All tests passed.")



ValueError: f(a) and f(b) must have different signs